# 03 – Group-Aware Validation and Catalog Diagnostics

Compare random row-level cross-validation with host-star-grouped cross-validation. 
Only confirmed and false-positive rows are used as labeled training examples; 
candidate rows remain unlabeled. This notebook reports descriptive catalog 
fractions and out-of-fold classification performance, **not** a completeness-
corrected astrophysical occurrence rate.


In [1]:
# CELL 1 — Load the catalog, define labels/features, and create leakage-safe OOF predictions

import os
import glob
import numpy as np
import pandas as pd

from sklearn.model_selection import StratifiedKFold, StratifiedGroupKFold
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier

RANDOM_STATE = 42
N_SPLITS = 5
THRESHOLD = 0.50  # Fixed in advance; do not tune it on these same OOF results.

# Kaggle path handling: auto-detect a single likely KOI cumulative CSV.
# If multiple files are found, set DATA_PATH manually to the intended file.
DATA_PATH = None  # Example: "/kaggle/input/your-dataset/cumulative.csv"

if DATA_PATH is None:
    csv_candidates = glob.glob("/kaggle/input/**/*.csv", recursive=True)
    koi_candidates = [
        p for p in csv_candidates
        if "cumulative" in os.path.basename(p).lower()
        or "koi" in os.path.basename(p).lower()
    ]
    if len(koi_candidates) == 1:
        DATA_PATH = koi_candidates[0]
    elif len(koi_candidates) == 0:
        raise FileNotFoundError(
            "No likely KOI cumulative CSV found under /kaggle/input. "
            "Attach the dataset and set DATA_PATH manually."
        )
    else:
        print("Possible KOI CSV files:")
        for path in koi_candidates:
            print(path)
        raise ValueError(
            "Multiple likely KOI CSV files were found. Set DATA_PATH manually "
            "to the exact cumulative catalog used in Notebook 02."
        )

print("Using catalog:", DATA_PATH)
raw = pd.read_csv(DATA_PATH, comment="#", low_memory=False)

required = {"kepid", "koi_disposition", "koi_prad", "koi_steff"}
missing = required - set(raw.columns)
if missing:
    raise KeyError(f"Required columns missing from catalog: {sorted(missing)}")

for col in ["kepid", "koi_prad", "koi_steff"]:
    raw[col] = pd.to_numeric(raw[col], errors="coerce")

# Match Notebook 02's sample definition exactly.
catalog = raw[
    raw["koi_disposition"].isin(["CONFIRMED", "FALSE POSITIVE", "CANDIDATE"])
    & raw["kepid"].notna()
    & raw["koi_prad"].between(1.0, 4.0, inclusive="both")
    & raw["koi_steff"].between(4700, 6500, inclusive="both")
].copy()

catalog["group"] = catalog["kepid"].astype("int64")

# Only resolved dispositions are supervised labels. Candidates are NOT negatives.
labeled = catalog[
    catalog["koi_disposition"].isin(["CONFIRMED", "FALSE POSITIVE"])
].copy()
labeled["label"] = (labeled["koi_disposition"] == "CONFIRMED").astype(int)

# Exclude direct disposition/reliability fields (e.g., koi_score and vetting flags).
# These candidate predictors still require scientific review before final claims.
candidate_features = [
    "koi_period", "koi_time0bk", "koi_impact", "koi_duration",
    "koi_depth", "koi_prad", "koi_teq", "koi_insol",
    "koi_steff", "koi_slogg", "koi_srad", "koi_kepmag"
]
feature_cols = [c for c in candidate_features if c in labeled.columns]
if len(feature_cols) < 3:
    raise ValueError(
        "Fewer than 3 candidate features were found. Inspect the CSV columns "
        "and revise candidate_features deliberately."
    )

X = labeled[feature_cols].apply(pd.to_numeric, errors="coerce")
y = labeled["label"].to_numpy()
groups = labeled["group"].to_numpy()

# Remove unusable predictors before cross-validation.
drop_cols = [
    c for c in X.columns
    if X[c].notna().sum() == 0 or X[c].nunique(dropna=True) <= 1
]
if drop_cols:
    print("Dropping all-missing/constant features:", drop_cols)
    X = X.drop(columns=drop_cols)
    feature_cols = X.columns.tolist()
if X.shape[1] == 0:
    raise ValueError("No usable features remain after feature checks.")
if len(np.unique(y)) != 2:
    raise ValueError("Both CONFIRMED and FALSE POSITIVE labels are required.")

print("Filtered catalog rows:", len(catalog))
print("Unique host stars in catalog:", catalog["group"].nunique())
print("Resolved labeled rows:", len(labeled))
print("Unique host stars in labeled data:", labeled["group"].nunique())
print("Class counts (0=false positive, 1=confirmed):")
print(pd.Series(y).value_counts().sort_index())
print("Features:", feature_cols)

# Each class must occur across enough distinct host stars for grouped folds.
groups_per_class = labeled.groupby("label")["group"].nunique()
print("Distinct host stars by class:")
print(groups_per_class)
if groups_per_class.min() < N_SPLITS:
    raise ValueError(
        f"Cannot use {N_SPLITS} stratified group folds: at least one class has "
        "too few distinct host stars. Choose a smaller N_SPLITS only after "
        "checking the fold class distributions."
    )

def build_model():
    # Imputation is inside the pipeline, so medians are learned on each training fold only.
    return Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("classifier", RandomForestClassifier(
            n_estimators=400,
            min_samples_leaf=2,
            class_weight="balanced",
            random_state=RANDOM_STATE,
            n_jobs=-1
        ))
    ])

def make_oof_predictions(splitter, X, y, groups=None, grouped=False):
    probabilities = np.full(len(y), np.nan, dtype=float)
    fold_ids = np.full(len(y), -1, dtype=int)

    splits = splitter.split(X, y, groups) if grouped else splitter.split(X, y)
    for fold_id, (train_idx, test_idx) in enumerate(splits):
        if np.any(fold_ids[test_idx] != -1):
            raise RuntimeError("A row was assigned to more than one test fold.")

        if grouped:
            train_groups = set(groups[train_idx])
            test_groups = set(groups[test_idx])
            if train_groups.intersection(test_groups):
                raise RuntimeError("Host-star overlap found between train and test.")

        if len(np.unique(y[train_idx])) != 2:
            raise ValueError(f"Training fold {fold_id + 1} contains only one class.")
        if len(np.unique(y[test_idx])) != 2:
            raise ValueError(
                f"Test fold {fold_id + 1} contains only one class; pooled ROC-AUC "
                "would still be computable, but fold diagnostics are unstable. "
                "Inspect the group distribution or revise the number of folds."
            )

        model = build_model()
        model.fit(X.iloc[train_idx], y[train_idx])
        probabilities[test_idx] = model.predict_proba(X.iloc[test_idx])[:, 1]
        fold_ids[test_idx] = fold_id

    if np.isnan(probabilities).any() or np.any(fold_ids == -1):
        raise RuntimeError("Some labeled rows did not receive exactly one OOF prediction.")
    return probabilities, fold_ids

random_cv = StratifiedKFold(
    n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE
)
group_cv = StratifiedGroupKFold(
    n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE
)

random_oof, random_fold = make_oof_predictions(
    random_cv, X, y, grouped=False
)
grouped_oof, grouped_fold = make_oof_predictions(
    group_cv, X, y, groups=groups, grouped=True
)

print("Completed exactly one OOF prediction per resolved labeled row.")
print("Random fold counts:", np.bincount(random_fold))
print("Grouped fold counts:", np.bincount(grouped_fold))


Using catalog: /kaggle/input/datasets/sairakeshnagaram/koi-dataset/cumulative_2026.10.20_19.25.07.csv
Filtered catalog rows: 4249
Unique host stars in catalog: 3517
Resolved labeled rows: 3195
Unique host stars in labeled data: 2654
Class counts (0=false positive, 1=confirmed):
0    1229
1    1966
Name: count, dtype: int64
Features: ['koi_period', 'koi_time0bk', 'koi_impact', 'koi_duration', 'koi_depth', 'koi_prad', 'koi_teq', 'koi_insol', 'koi_steff', 'koi_slogg', 'koi_srad', 'koi_kepmag']
Distinct host stars by class:
label
0    1209
1    1463
Name: group, dtype: int64
Completed exactly one OOF prediction per resolved labeled row.
Random fold counts: [639 639 639 639 639]
Grouped fold counts: [628 650 660 628 629]


In [2]:
# CELL 2 — Compare out-of-fold validation performance

from sklearn.metrics import (
    roc_auc_score, average_precision_score, balanced_accuracy_score,
    brier_score_loss, precision_score, recall_score, f1_score
)


def summarize_metrics(name, y_true, probability, threshold=THRESHOLD):
    predicted = (probability >= threshold).astype(int)
    return {
        "validation": name,
        "ROC_AUC": roc_auc_score(y_true, probability),
        "average_precision": average_precision_score(y_true, probability),
        "balanced_accuracy": balanced_accuracy_score(y_true, predicted),
        "Brier_score_lower_is_better": brier_score_loss(y_true, probability),
        "precision_at_fixed_threshold": precision_score(
            y_true, predicted, zero_division=0
        ),
        "recall_at_fixed_threshold": recall_score(
            y_true, predicted, zero_division=0
        ),
        "F1_at_fixed_threshold": f1_score(
            y_true, predicted, zero_division=0
        ),
        "threshold": threshold
    }

metrics = pd.DataFrame([
    summarize_metrics("Random row-level CV", y, random_oof),
    summarize_metrics("Host-star grouped CV", y, grouped_oof)
])

print("=== POOLED OUT-OF-FOLD VALIDATION COMPARISON ===")
display(metrics)

# Explicitly verify the grouped split has no host-star overlap.
print("\nGrouped validation integrity check:")
for fold_id in range(N_SPLITS):
    test_groups = set(groups[grouped_fold == fold_id])
    train_groups = set(groups[grouped_fold != fold_id])
    overlap = test_groups.intersection(train_groups)
    print(f"Fold {fold_id + 1}: host-star overlap = {len(overlap)}")
    if overlap:
        raise RuntimeError("Grouped cross-validation integrity check failed.")


=== POOLED OUT-OF-FOLD VALIDATION COMPARISON ===


,validation,ROC_AUC,average_precision,balanced_accuracy,Brier_score_lower_is_better,precision_at_fixed_threshold,recall_at_fixed_threshold,F1_at_fixed_threshold,threshold
0,Random row-level CV,0.926194,0.944526,0.847124,0.103644,0.867146,0.919634,0.892619,0.5
1,Host-star grouped CV,0.923559,0.943764,0.845038,0.105693,0.867569,0.913021,0.889715,0.5



Grouped validation integrity check:
Fold 1: host-star overlap = 0
Fold 2: host-star overlap = 0
Fold 3: host-star overlap = 0
Fold 4: host-star overlap = 0
Fold 5: host-star overlap = 0


In [3]:
# CELL 3 — Descriptive catalog fraction and export OOF scores

# Match Notebook 02's descriptive catalog fraction exactly.
# This is NOT a completeness-corrected planet occurrence-rate estimate.
n_catalog_stars = catalog["group"].nunique()
confirmed_host_stars = catalog.loc[
    catalog["koi_disposition"].eq("CONFIRMED"), "group"
].nunique()

if n_catalog_stars == 0:
    raise ValueError("No stars remain after the catalog filters.")

baseline_catalog_fraction = confirmed_host_stars / n_catalog_stars

summary = pd.DataFrame([{
    "filtered_catalog_rows": len(catalog),
    "unique_stars_in_filtered_catalog": n_catalog_stars,
    "stars_with_at_least_one_confirmed_entry": confirmed_host_stars,
    "baseline_confirmed_catalog_fraction": baseline_catalog_fraction,
    "baseline_confirmed_catalog_fraction_percent": 100 * baseline_catalog_fraction,
    "radius_min_earth": 1.0,
    "radius_max_earth": 4.0,
    "stellar_temperature_min_K": 4700,
    "stellar_temperature_max_K": 6500,
    "interpretation": (
        "Descriptive fraction of stars represented in this filtered KOI catalog "
        "with at least one confirmed entry; not a completeness-corrected "
        "astrophysical occurrence rate."
    )
}])

print("=== DESCRIPTIVE CATALOG SUMMARY ===")
display(summary)

# Export the catalog with OOF probabilities only for rows with resolved labels.
# Candidate rows deliberately retain NaN probabilities; they are not assigned labels.
scored_catalog = catalog.copy()
scored_catalog["random_oof_probability"] = np.nan
scored_catalog["grouped_oof_probability"] = np.nan
scored_catalog["random_oof_fold"] = -1
scored_catalog["grouped_oof_fold"] = -1

labeled_index = labeled.index
scored_catalog.loc[labeled_index, "random_oof_probability"] = random_oof
scored_catalog.loc[labeled_index, "grouped_oof_probability"] = grouped_oof
scored_catalog.loc[labeled_index, "random_oof_fold"] = random_fold
scored_catalog.loc[labeled_index, "grouped_oof_fold"] = grouped_fold

# This flag is a thresholded classifier output for labeled rows only, not a
# candidate disposition and not an occurrence-rate correction.
scored_catalog["grouped_oof_predicted_confirmed_at_threshold"] = np.nan
scored_catalog.loc[labeled_index,
    "grouped_oof_predicted_confirmed_at_threshold"] = (
        grouped_oof >= THRESHOLD
    ).astype(int)

os.makedirs("/kaggle/working", exist_ok=True)
metrics.to_csv("/kaggle/working/03_validation_metrics.csv", index=False)
summary.to_csv("/kaggle/working/03_catalog_fraction_summary.csv", index=False)
scored_catalog.to_csv("/kaggle/working/03_scored_catalog_oof.csv", index=False)

oof = labeled[["kepid", "koi_disposition", "label", "group"]].copy()
oof["random_oof_probability"] = random_oof
oof["random_fold"] = random_fold
oof["grouped_oof_probability"] = grouped_oof
oof["grouped_fold"] = grouped_fold
oof.to_csv("/kaggle/working/03_oof_predictions.csv", index=False)

print("Saved outputs in /kaggle/working:")
for filename in [
    "03_validation_metrics.csv",
    "03_catalog_fraction_summary.csv",
    "03_scored_catalog_oof.csv",
    "03_oof_predictions.csv"
]:
    print(" -", filename)

print("\nSCIENTIFIC INTERPRETATION:")
print(
    "The random-versus-grouped comparison evaluates classification performance "
    "under two validation designs. The catalog fraction is a descriptive "
    "catalog statistic. Neither result, by itself, estimates the intrinsic "
    "occurrence rate of planets. Candidates remain unlabeled and receive no "
    "supervised prediction in this notebook."
)


=== DESCRIPTIVE CATALOG SUMMARY ===


,filtered_catalog_rows,unique_stars_in_filtered_catalog,stars_with_at_least_one_confirmed_entry,baseline_confirmed_catalog_fraction,baseline_confirmed_catalog_fraction_percent,radius_min_earth,radius_max_earth,stellar_temperature_min_K,stellar_temperature_max_K,interpretation
0,4249,3517,1463,0.41598,41.597953,1.0,4.0,4700,6500,Descriptive fraction of stars represented in t...


Saved outputs in /kaggle/working:
 - 03_validation_metrics.csv
 - 03_catalog_fraction_summary.csv
 - 03_scored_catalog_oof.csv
 - 03_oof_predictions.csv

SCIENTIFIC INTERPRETATION:
The random-versus-grouped comparison evaluates classification performance under two validation designs. The catalog fraction is a descriptive catalog statistic. Neither result, by itself, estimates the intrinsic occurrence rate of planets. Candidates remain unlabeled and receive no supervised prediction in this notebook.


In [4]:
# CELL 4 — List generated outputs

import os

print("Files in /kaggle/working:")
for filename in sorted(os.listdir("/kaggle/working")):
    if filename.startswith("03_"):
        print(os.path.join("/kaggle/working", filename))


Files in /kaggle/working:
/kaggle/working/03_catalog_fraction_summary.csv
/kaggle/working/03_oof_predictions.csv
/kaggle/working/03_scored_catalog_oof.csv
/kaggle/working/03_validation_metrics.csv


In [5]:
# CELL 5 — Optional: bundle Notebook 03 CSV outputs into one folder

import os
import shutil

output_dir = "/kaggle/working/notebook3_results"
os.makedirs(output_dir, exist_ok=True)

files_to_copy = [
    "03_validation_metrics.csv",
    "03_catalog_fraction_summary.csv",
    "03_scored_catalog_oof.csv",
    "03_oof_predictions.csv"
]

for filename in files_to_copy:
    source_path = os.path.join("/kaggle/working", filename)
    if os.path.exists(source_path):
        shutil.copy2(source_path, os.path.join(output_dir, filename))
    else:
        raise FileNotFoundError(f"Expected output not found: {source_path}")

print("Prepared files:")
for filename in sorted(os.listdir(output_dir)):
    print(os.path.join(output_dir, filename))


Prepared files:
/kaggle/working/notebook3_results/03_catalog_fraction_summary.csv
/kaggle/working/notebook3_results/03_oof_predictions.csv
/kaggle/working/notebook3_results/03_scored_catalog_oof.csv
/kaggle/working/notebook3_results/03_validation_metrics.csv
